# 🐘 Детекция объектов: African Wildlife (YOLO11n, Ultralytics)
**CSCI1409 Deep Learning — Bonus Project.**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_USERNAME/YOUR_REPO/blob/main/object_detection.ipynb)

Ноутбук запускается **с чистого рантайма** сверху вниз и делает всё:
1. ставит зависимости с зафиксированными версиями;
2. скачивает датасет, проверяет разметку и ищет утечки между train/val/test;
3. обучает базовую модель (baseline) и модель эксперимента (меняется **один** параметр);
4. сравнивает их на **validation**;
5. выбирает финальную модель по validation и **один раз** оценивает её на **test**;
6. показывает предсказания и разбирает три случая ошибок.

**Перед запуском:** `Runtime → Change runtime type → T4 GPU`. Замени `YOUR_USERNAME/YOUR_REPO` в ссылке выше и в ячейке клонирования на свои.

## 0. Окружение
Проверяем, что есть GPU. Обучение на GPU занимает минуты, на CPU — часы.

In [ ]:
import sys, torch
print("Python:", sys.version.split()[0], "| torch:", torch.__version__)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("⚠️ GPU не найден: Runtime → Change runtime type → T4 GPU. На CPU обучение будет очень медленным.")

## 1. Получаем код репозитория
Ноутбук использует скрипты из папки `scripts/` и конфиги из `configs/`, поэтому клонируем репозиторий. **Токены и пароли в ноутбук не вставляем** — репозиторий должен быть доступен преподавателю, а для клонирования достаточно публичной ссылки (или добавь преподавателя и ассистентов как collaborators).

In [ ]:
import os
REPO_URL = "https://github.com/YOUR_USERNAME/YOUR_REPO.git"   # <-- ЗАМЕНИ на ссылку на свой репозиторий
REPO_DIR = "/content/detector"
if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}

## 2. Зависимости (зафиксированные версии)
`requirements.txt` содержит `ultralytics==8.4.199` (точная версия). Остальные пакеты в Colab уже есть. Точные версии **этого** прогона сохраняем командой `pip freeze` — это и есть доказательство воспроизводимости.

> Если Colab попросит перезапустить среду после установки — перезапусти и выполни ячейки заново, начиная с ячейки `%cd`.

In [ ]:
!pip install -q -r requirements.txt
os.makedirs("results/figures", exist_ok=True)
os.makedirs("weights", exist_ok=True)
!pip freeze > results/environment_freeze.txt
import ultralytics
print("ultralytics:", ultralytics.__version__)

## 3. Импорты, конфигурация и seed
Все настройки обучения лежат в `configs/experiments.yaml` — единый источник правды. Мы фиксируем `seed`, чтобы прогон можно было повторить (полная детерминированность на GPU не гарантируется — небольшие расхождения метрик возможны, об этом написано в README).

In [ ]:
import json, time, random, shutil
from pathlib import Path
import numpy as np, pandas as pd, yaml
from IPython.display import display, Image as IPImage

sys.path.insert(0, os.getcwd())
from scripts.common import list_images, load_data_cfg
from scripts.visualize import show_ground_truth, show_predictions, show_failures, plot_curves
from scripts.failure_analysis import evaluate_images, worst_images
from ultralytics import YOLO

CFG = yaml.safe_load(open("configs/experiments.yaml"))
EV, SEED = CFG["eval"], CFG["seed"]
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
PROJECT = str(Path("results/runs").resolve())
print(json.dumps(CFG, indent=2, ensure_ascii=False))

## 4. Датасет: загрузка, проверка, статистика
**Датасет:** Ultralytics *African Wildlife* (автор оригинала — Bianca Ferreira, адаптация Ultralytics): 1 504 изображения, 4 класса — `buffalo, elephant, rhino, zebra`. Разметка — **YOLO txt**: в каждой строке `class x_center y_center width height`, координаты нормированы в диапазон 0–1. Разбиение готовое: train 1052 / val 225 / test 227.

Скрипт `scripts/prepare_data.py`:
* скачивает датасет (~100 МБ);
* считает изображения и объекты по классам в каждой выборке;
* **проверяет разметку**: битые строки, координаты вне [0,1], нулевые боксы, повторяющиеся боксы, неверные номера классов, изображения без файла разметки;
* **ищет утечки** между выборками: точные копии (md5) и почти-копии (перцептивный хэш dHash, расстояние Хэмминга ≤ 4). Если найдена картинка из train, похожая на картинку из val/test, она **удаляется из train** (val/test остаются нетронутыми), список записывается в `configs/train_dedup.txt`.

In [ ]:
!python scripts/prepare_data.py
DATA_YAML = str(Path("configs/data.yaml").resolve())
DATA = load_data_cfg(DATA_YAML)
NAMES = [DATA["names"][i] for i in sorted(DATA["names"])]
report = json.load(open("results/dataset_report.json"))

def split_table(key):
    rows = {}
    for s, st in report[key].items():
        rows[s] = {"images": st["images"], "objects": st["objects"], **st["per_class"],
                   "empty labels": st["background_images_empty_label"],
                   "no label file": st["images_without_label_file"],
                   "label problems": sum(st["label_problems"].values()),
                   "median box area (доля кадра)": round(st["median_box_area_fraction"] or 0, 3)}
    return pd.DataFrame(rows).T

print("=== После очистки от утечек (используется для обучения) ===")
display(split_table("splits_final"))
print("Утечек между выборками:", report["leaks_cross_split_count"], "| удалено:", report["removed_images"])
for s, st in report["splits_final"].items():
    if st["problem_examples"]:
        print(f"Проблемы разметки в {s}:", st["problem_examples"][:5])

### 4.1 Ground truth: изображения с реальными рамками
Смотрим глазами, что рамки лежат на объектах и классы правильные. Это часть проверки разметки (формальные проверки не заменяют просмотр).

In [ ]:
train_imgs = list_images(DATA["train"])
_ = show_ground_truth(train_imgs, NAMES, n=8, seed=SEED, save="results/figures/ground_truth_train.png")

## 5. Гипотеза эксперимента (записана ДО получения результатов)
**Меняем один параметр:** размер входного изображения `imgsz`: 640 (baseline) → 320 (эксперимент). Данные, seed, стартовые веса `yolo11n.pt`, оптимизатор, lr, число эпох, batch — одинаковые.

**Гипотеза H1.** Уменьшение `imgsz` вдвое по каждой стороне даёт в ~4 раза меньше пикселей, поэтому обучение ускорится в 2–4 раза. Так как животные в датасете в основном крупные (см. медианную долю площади бокса в таблице выше), mAP@0.50 снизится незначительно (ожидание: не более ~3 п.п.), а потеря качества будет сосредоточена в мелких/далёких животных (снизится recall).

> ✍️ **Перепиши гипотезу своими словами и при желании измени числа — до запуска обучения!** Эксперимент считается валидным, если гипотеза зафиксирована заранее; подтвердится она или нет — не важно.

## 6. Обучение
Функция `run_training` берёт общие настройки из `common`, добавляет изменения из `runs.<имя>` и обучает **с одних и тех же стартовых весов** `yolo11n.pt` (дообучение — *fine-tuning*: предобученная на COCO сеть уже умеет находить края, текстуры, животных; мы подстраиваем её под 4 класса).

* Лучший чекпоинт `best.pt` выбирается Ultralytics **по validation** (fitness = 0.1·mAP@0.50 + 0.9·mAP@0.50-0.95 на val), test в обучении не участвует.
* Логи, `args.yaml`, `results.csv` и графики сохраняются в `results/runs/<имя>/`.
* Время обучения измеряем вручную.

In [ ]:
runs = {}

def run_training(run_name):
    overrides = CFG["runs"][run_name] or {}
    args = {**CFG["common"], **overrides}
    model = YOLO(CFG["model"])                       # одинаковые начальные веса для обоих запусков
    t0 = time.time()
    model.train(data=DATA_YAML, project=PROJECT, name=run_name, exist_ok=True, seed=SEED, **args)
    minutes = (time.time() - t0) / 60
    save_dir = Path(model.trainer.save_dir)
    runs[run_name] = dict(name=run_name, overrides=overrides, args=args, seed=SEED,
                          framework=f"ultralytics {ultralytics.__version__}, torch {torch.__version__}",
                          initial_weights=CFG["model"], train_minutes=round(minutes, 2),
                          save_dir=str(save_dir), best=str(save_dir / "weights" / "best.pt"))
    print(f"\n{run_name}: обучение заняло {minutes:.1f} мин; best = {runs[run_name]['best']}")
    return runs[run_name]

### 6.1 Baseline (imgsz=640)

In [ ]:
run_training("baseline_640")

### 6.2 Эксперимент (imgsz=320)
Проверка «контролируемости»: в блоке эксперимента должен быть ровно один изменённый параметр.

In [ ]:
assert len(CFG["runs"]["exp_imgsz320"]) == 1, "В контролируемом эксперименте меняется ровно один параметр!"
print("Изменяется только:", CFG["runs"]["exp_imgsz320"])
run_training("exp_imgsz320")

## 7. Сравнение на validation
Метрики (подробнее — в README и `DEFENSE_PREP.md`):
* **Precision** — доля верных рамок среди всех предсказанных (насколько можно доверять срабатываниям);
* **Recall** — доля найденных объектов среди всех реальных (сколько ничего не пропущено);
* **mAP@0.50** — средняя по классам площадь под кривой precision–recall, где рамка считается верной при IoU ≥ 0.5.

Настройки оценки: `conf=0.001` (почти все рамки сохраняются, чтобы построить полную PR-кривую), NMS IoU = 0.7, порог совпадения для mAP@0.50 = 0.5. Precision/recall, которые выдаёт Ultralytics, считаются в точке порога уверенности, максимизирующей F1.

In [ ]:
def evaluate(weights, split, imgsz, name):
    m = YOLO(weights)
    r = m.val(data=DATA_YAML, split=split, imgsz=imgsz, batch=16, conf=EV["val_conf"], iou=EV["nms_iou"],
              project=PROJECT, name=name, exist_ok=True, plots=True)
    d = r.results_dict
    out = {"precision": float(d["metrics/precision(B)"]), "recall": float(d["metrics/recall(B)"]),
           "mAP50": float(d["metrics/mAP50(B)"]), "mAP50-95": float(d["metrics/mAP50-95(B)"]),
           "speed_ms_per_image": {k: round(float(v), 2) for k, v in r.speed.items()},
           "settings": {"split": split, "imgsz": imgsz, "conf": EV["val_conf"],
                        "nms_iou": EV["nms_iou"], "map_iou": EV["map_iou"]}}
    try:
        out["per_class_AP50"] = {r.names[int(c)]: round(float(a), 4) for c, a in zip(r.box.ap_class_index, r.box.ap50)}
    except Exception as e:
        print("per-class AP недоступен:", e)
    return out

for n, info in runs.items():
    info["val"] = evaluate(info["best"], "val", info["args"]["imgsz"], f"val_{n}")
    json.dump(info, open(f"results/{n}_summary.json", "w"), indent=2, ensure_ascii=False)

rows = []
for n, info in runs.items():
    v = info["val"]
    rows.append({"run": n,
                 "changed setting": ", ".join(f"{k}={x}" for k, x in info["overrides"].items()) or "— (baseline)",
                 "precision": round(v["precision"], 4), "recall": round(v["recall"], 4),
                 "mAP@0.50": round(v["mAP50"], 4), "mAP@0.50-0.95": round(v["mAP50-95"], 4),
                 "train time, min": info["train_minutes"],
                 "inference, ms/img": v["speed_ms_per_image"].get("inference")})
cmp = pd.DataFrame(rows)
cmp.to_csv("results/experiment_comparison.csv", index=False)
display(cmp)
for n, info in runs.items():
    print(n, "AP@0.50 по классам:", info["val"].get("per_class_AP50"))

### 7.1 Кривые обучения

In [ ]:
plot_curves({n: i["save_dir"] for n, i in runs.items()}, save="results/figures/learning_curves_compare.png")
for n, i in runs.items():
    print(n); display(IPImage(str(Path(i["save_dir"]) / "results.png")))

### 7.2 ✍️ Объяснение результата эксперимента (заполни своими словами)
Сравни с гипотезой из раздела 5. Подтвердилась ли она? Что произошло с временем, precision, recall и mAP@0.50? Какой класс пострадал сильнее (см. AP по классам)? Почему (размер объектов, потеря деталей при уменьшении картинки)? Какова погрешность сравнения — один seed, разница в 1 п.п. может быть шумом.

## 8. Выбор финальной модели (только по validation!)
Правило выбора: финальной становится запуск с **наибольшим val mAP@0.50**. Test в выборе не участвует. Лучший чекпоинт копируем в `weights/final_best.pt` (файл ~5 МБ, его можно положить прямо в GitHub).

In [ ]:
final_name = max(runs, key=lambda n: runs[n]["val"]["mAP50"])
FINAL = runs[final_name]
IMGSZ = FINAL["args"]["imgsz"]
shutil.copy(FINAL["best"], "weights/final_best.pt")
print("Финальная модель (по val mAP@0.50):", final_name, "| imgsz =", IMGSZ)

## 9. Финальная оценка на TEST (один раз)
Считаем два набора чисел:
1. **Официальные метрики Ultralytics** (`conf=0.001`, NMS IoU=0.7, mAP@0.50);
2. **Метрики в фиксированной рабочей точке** `conf=0.25`, совпадение рамки при IoU ≥ 0.5 — собственная проверка (`scripts/failure_analysis.py`). Здесь порог выбран заранее, а не подогнан под выборку, поэтому такие precision/recall честнее отражают, как модель поведёт себя при реальном использовании.

In [ ]:
test_official = evaluate("weights/final_best.pt", "test", IMGSZ, f"test_{final_name}")
final_model = YOLO("weights/final_best.pt")
test_imgs = list_images(DATA["test"])
records, fixed = evaluate_images(final_model, test_imgs, conf=EV["fixed_conf"], nms_iou=EV["nms_iou"],
                                 imgsz=IMGSZ, match_iou=EV["fixed_match_iou"])
test_result = {"final_run": final_name, "weights": "weights/final_best.pt", "imgsz": IMGSZ,
               "official_ultralytics": test_official, "fixed_threshold": fixed}
json.dump(test_result, open("results/test_metrics.json", "w"), indent=2, ensure_ascii=False)
pd.DataFrame(records).to_csv("results/test_per_image_errors.csv", index=False)

display(pd.DataFrame([
    {"метод": f"Ultralytics (conf={EV['val_conf']}, NMS IoU={EV['nms_iou']}, mAP IoU={EV['map_iou']})",
     "precision": round(test_official["precision"], 4), "recall": round(test_official["recall"], 4),
     "mAP@0.50": round(test_official["mAP50"], 4)},
    {"метод": f"Фикс. порог (conf={EV['fixed_conf']}, match IoU={EV['fixed_match_iou']})",
     "precision": round(fixed["precision"], 4), "recall": round(fixed["recall"], 4), "mAP@0.50": "—"}]))
print("AP@0.50 по классам на test:", test_official.get("per_class_AP50"))
print("Типы ошибок при conf=0.25:", {k: fixed[k] for k in ["tp","fn","fp_bg","fp_class","fp_loc","fp_dup"]})

### 9.1 Предсказания на test (рамка, класс, confidence)

In [ ]:
_ = show_predictions(final_model, test_imgs, n=8, seed=SEED, conf=EV["fixed_conf"], iou=EV["nms_iou"],
                     imgsz=IMGSZ, save="results/figures/test_predictions.png")

## 10. Разбор ошибок (failure analysis)
Берём 3 тестовых изображения с наибольшим числом ошибок. Типы ошибок:
* `fn` — пропущенный объект; `fp_bg` — ложное срабатывание на фоне; `fp_class` — рамка на объекте, но не тот класс; `fp_loc` — класс верный, рамка неточная; `fp_dup` — дубликат рамки.

Это **анализ после** выбора модели — на настройки обучения он не влияет (test не используется для выбора).

In [ ]:
worst = worst_images(records, k=3)
display(pd.DataFrame(worst)[["image", "n_gt", "n_pred", "tp", "fn", "fp_bg", "fp_class", "fp_loc", "fp_dup", "errors"]])
show_failures(final_model, worst, NAMES, conf=EV["fixed_conf"], iou=EV["nms_iou"], imgsz=IMGSZ,
              save="results/figures/failure_cases.png")

### 10.1 ✍️ Анализ трёх случаев (заполни своими словами — скопируй в README)
Для каждого из трёх изображений: **что именно пошло не так** (пропуск / ложная рамка / не тот класс / неточная рамка), **вероятная причина** (перекрытие животных, мелкий/далёкий объект, плохой свет, похожие классы — например, носорог и буйвол, обрезанный объект, неоднозначная разметка), и **один практический следующий шаг** (например: больше данных с такими сценами, увеличить imgsz, более сильная модель `yolo11s`, аугментации, пересмотр разметки). Если ошибок меньше трёх — так и напиши и разбери самые сложные примеры.

## 11. Сохранение результатов
Скачай архив с результатами и положи его содержимое в репозиторий (папки `results/`, `weights/`, `configs/`). Сохрани этот ноутбук **вместе с выводами ячеек**: `File → Save a copy in GitHub`. Ссылку на `weights/final_best.pt` (`https://github.com/<USER>/<REPO>/raw/main/weights/final_best.pt`) укажи в README.

In [ ]:
!zip -r -q /content/submission_artifacts.zip results weights configs
try:
    from google.colab import files
    files.download("/content/submission_artifacts.zip")
except Exception as e:
    print("Архив: /content/submission_artifacts.zip", e)